# Proyecto final MEMAD – Semana 2
## Comparación de optimizadores entrenando una CNN desde cero

**Objetivo de la semana:** comparar los 7 optimizadores implementados en la semana 1 (SGD, momento,
Nesterov, Adagrad, RMSprop, Adam y AdamW) entrenando la misma red **sin pesos preentrenados**.

**Protocolo (igual para todos los optimizadores):**

| Elemento | Valor |
|---|---|
| Red | ResNet-18 sin preentrenar, entrada de 1 canal (escala de grises) |
| Imágenes | 128 × 128, normalizadas con la media y desviación del conjunto de entrenamiento |
| Partición | La misma de la semana 1: 4 760 entrenamiento / 840 validación / 1 600 prueba |
| Aumento de datos | Volteo horizontal y desplazamiento aleatorio de ±8 píxeles |
| Presupuesto | 15 épocas, lotes de 64, tasa de aprendizaje **constante** |
| Tasa de aprendizaje | Elegida para cada optimizador con el *LR range test* (Smith, 2017) |
| Repeticiones | 3 semillas por optimizador → se reporta media ± desviación estándar |
| Selección del modelo | Mejor época según **validación**; el conjunto de prueba se evalúa una sola vez al final |

**Entregables que genera este notebook:**
1. Código del marco de experimentos (este notebook).
2. Curvas de pérdida y exactitud por época de cada optimizador.
3. Tabla de resultados: media ± desviación estándar en validación y prueba, y tiempo por época.

**Cómo ejecutarlo:**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución →* **GPU** (T4 basta; L4 es más rápida).
2. En los *Secrets* de Colab (icono de llave) deben existir `KAGGLE_USERNAME` y `KAGGLE_KEY`.
3. *Entorno de ejecución → Ejecutar todas*. Tarda aproximadamente **30–40 min en T4** (≈ 20 min en L4).

Si Colab se desconecta a mitad, basta con volver a ejecutar todo: los experimentos ya terminados se
leen de `resultados_semana2/` y no se repiten (activa `USAR_DRIVE = True` para que esos resultados
sobrevivan aunque se reinicie el entorno).

## 0. Configuración

In [ ]:
import os, time, math, json, copy, random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

# ---- Parámetros del experimento -------------------------------------------
IMG_SIZE    = 128
BATCH_SIZE  = 64
EPOCHS      = 15
SEEDS       = [0, 1, 2]
VAL_FRAC    = 0.15
SPLIT_SEED  = 42          # misma partición que en la semana 1
RANGE_ITERS = 200         # iteraciones del LR range test
DATA_DIR    = './brain_tumor_data'

# Guardar resultados en Google Drive para no perderlos si se reinicia el entorno
USAR_DRIVE = False
OUT = 'resultados_semana2'
if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/MEMAD_semana2'
os.makedirs(OUT, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
print('Dispositivo:', device, '|', torch.cuda.get_device_name(0) if USE_AMP else 'SIN GPU')
if not USE_AMP:
    print('AVISO: sin GPU el notebook tardará horas. Cambia el entorno de ejecución a GPU.')

## 1. Descarga del dataset

In [ ]:
if not os.path.isdir(os.path.join(DATA_DIR, 'Training')):
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    except Exception as e:
        print('No se pudieron leer los Secrets de Kaggle:', e)
    !pip install -q kaggle
    !kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset --unzip -p {DATA_DIR}
print('Carpetas:', os.listdir(DATA_DIR))

## 2. Optimizadores propios (semana 1)

Se usan **nuestras implementaciones**, ya validadas contra `torch.optim` en la semana 1.

In [ ]:
%%writefile optimizadores.py
# -*- coding: utf-8 -*-
"""
Implementación propia de algoritmos de optimización de primer orden.

Cada clase hereda de torch.optim.Optimizer y sigue las ecuaciones del
algoritmo original, con los mismos hiperparámetros por defecto que PyTorch
para poder validarlas contra torch.optim (ver test_optimizadores.py).

Notación: g_t = gradiente, theta = parámetros, eta = tasa de aprendizaje.
"""
import torch
from torch.optim import Optimizer


class SGDMomentum(Optimizer):
    """Descenso de gradiente estocástico con momento (heavy ball) o Nesterov.

        v_t     = mu * v_{t-1} + g_t                 (v_1 = g_1)
        clásico: theta_t = theta_{t-1} - eta * v_t
        Nesterov: theta_t = theta_{t-1} - eta * (g_t + mu * v_t)

    Con momentum=0 se reduce a SGD simple: theta_t = theta_{t-1} - eta * g_t.
    """

    def __init__(self, params, lr=1e-3, momentum=0.0, nesterov=False, weight_decay=0.0):
        if nesterov and momentum <= 0:
            raise ValueError("Nesterov requiere momentum > 0")
        super().__init__(params, dict(lr=lr, momentum=momentum, nesterov=nesterov,
                                      weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, mu, wd = group['lr'], group['momentum'], group['weight_decay']
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                if wd != 0:
                    g = g + wd * p                      # regularización L2
                if mu != 0:
                    state = self.state[p]
                    if 'v' not in state:
                        state['v'] = g.clone()
                    else:
                        state['v'].mul_(mu).add_(g)
                    v = state['v']
                    g = g + mu * v if group['nesterov'] else v
                p.add_(g, alpha=-lr)
        return loss


class Adagrad(Optimizer):
    """Adagrad (Duchi et al., 2011): tasa de aprendizaje adaptativa por parámetro.

        G_t     = G_{t-1} + g_t^2
        theta_t = theta_{t-1} - eta * g_t / (sqrt(G_t) + eps)
    """

    def __init__(self, params, lr=1e-2, eps=1e-10, weight_decay=0.0):
        super().__init__(params, dict(lr=lr, eps=eps, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                if group['weight_decay'] != 0:
                    g = g + group['weight_decay'] * p
                state = self.state[p]
                if 'G' not in state:
                    state['G'] = torch.zeros_like(p)
                state['G'].addcmul_(g, g)
                p.addcdiv_(g, state['G'].sqrt().add_(group['eps']), value=-group['lr'])
        return loss


class RMSprop(Optimizer):
    """RMSprop (Hinton, 2012): media móvil exponencial de g^2.

        s_t     = alpha * s_{t-1} + (1 - alpha) * g_t^2
        theta_t = theta_{t-1} - eta * g_t / (sqrt(s_t) + eps)
    """

    def __init__(self, params, lr=1e-2, alpha=0.99, eps=1e-8, weight_decay=0.0):
        super().__init__(params, dict(lr=lr, alpha=alpha, eps=eps, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            alpha = group['alpha']
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                if group['weight_decay'] != 0:
                    g = g + group['weight_decay'] * p
                state = self.state[p]
                if 's' not in state:
                    state['s'] = torch.zeros_like(p)
                state['s'].mul_(alpha).addcmul_(g, g, value=1 - alpha)
                p.addcdiv_(g, state['s'].sqrt().add_(group['eps']), value=-group['lr'])
        return loss


class Adam(Optimizer):
    """Adam (Kingma & Ba, 2015) y AdamW (Loshchilov & Hutter, 2019).

        m_t     = beta1 * m_{t-1} + (1 - beta1) * g_t
        v_t     = beta2 * v_{t-1} + (1 - beta2) * g_t^2
        m_hat   = m_t / (1 - beta1^t)          (corrección de sesgo)
        v_hat   = v_t / (1 - beta2^t)
        theta_t = theta_{t-1} - eta * m_hat / (sqrt(v_hat) + eps)

    decoupled=False: weight decay como término L2 en el gradiente (Adam).
    decoupled=True:  weight decay aplicado directamente a los pesos (AdamW):
                     theta_{t-1} <- theta_{t-1} * (1 - eta * lambda).
    """

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=0.0, decoupled=False):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps,
                                      weight_decay=weight_decay, decoupled=decoupled))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, (b1, b2), eps, wd = group['lr'], group['betas'], group['eps'], group['weight_decay']
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                if wd != 0:
                    if group['decoupled']:
                        p.mul_(1 - lr * wd)
                    else:
                        g = g + wd * p
                state = self.state[p]
                if 't' not in state:
                    state['t'] = 0
                    state['m'] = torch.zeros_like(p)
                    state['v'] = torch.zeros_like(p)
                state['t'] += 1
                t = state['t']
                state['m'].mul_(b1).add_(g, alpha=1 - b1)
                state['v'].mul_(b2).addcmul_(g, g, value=1 - b2)
                m_hat = state['m'] / (1 - b1 ** t)
                v_hat = state['v'] / (1 - b2 ** t)
                p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)
        return loss


def AdamW(params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-2):
    """AdamW = Adam con weight decay desacoplado (mismo default que torch.optim.AdamW)."""
    return Adam(params, lr=lr, betas=betas, eps=eps, weight_decay=weight_decay, decoupled=True)

In [ ]:
import importlib, optimizadores as opt
importlib.reload(opt)

# AdamW usa weight decay desacoplado (1e-2, valor por defecto de PyTorch);
# el resto se entrena sin weight decay para aislar el efecto del algoritmo.
OPTIMIZADORES = {
    'SGD':      lambda p, lr: opt.SGDMomentum(p, lr=lr),
    'Momento':  lambda p, lr: opt.SGDMomentum(p, lr=lr, momentum=0.9),
    'Nesterov': lambda p, lr: opt.SGDMomentum(p, lr=lr, momentum=0.9, nesterov=True),
    'Adagrad':  lambda p, lr: opt.Adagrad(p, lr=lr),
    'RMSprop':  lambda p, lr: opt.RMSprop(p, lr=lr),
    'Adam':     lambda p, lr: opt.Adam(p, lr=lr),
    'AdamW':    lambda p, lr: opt.AdamW(p, lr=lr, weight_decay=1e-2),
}
COLORES = dict(zip(OPTIMIZADORES, plt.cm.tab10.colors))

## 3. Carga de datos y partición

Las imágenes se cargan una sola vez en memoria de la GPU (≈ 120 MB a 128 × 128 en escala de grises), así
cada entrenamiento es rápido y no depende del `DataLoader`. La partición entrenamiento/validación es
**idéntica a la de la semana 1** (15 % estratificado, semilla 42).

In [ ]:
from PIL import Image

EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')

def cargar_carpeta(root):
    clases = sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)))
    X, y = [], []
    for ci, c in enumerate(clases):
        for f in sorted(os.listdir(os.path.join(root, c))):
            if f.lower().endswith(EXT):
                img = Image.open(os.path.join(root, c, f)).convert('L')
                X.append(np.asarray(img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.uint8))
                y.append(ci)
    return np.stack(X), np.array(y), clases

t0 = time.time()
X_full, y_full, CLASES = cargar_carpeta(os.path.join(DATA_DIR, 'Training'))
X_test, y_test, _ = cargar_carpeta(os.path.join(DATA_DIR, 'Testing'))

# Partición estratificada idéntica a la de la semana 1
np.random.seed(SPLIT_SEED)
tr_idx, va_idx = [], []
for c in np.unique(y_full):
    idx = np.where(y_full == c)[0]
    np.random.shuffle(idx)
    n_val = int(VAL_FRAC * len(idx))
    va_idx.extend(idx[:n_val]); tr_idx.extend(idx[n_val:])
tr_idx, va_idx = np.array(tr_idx), np.array(va_idx)

def a_gpu(X, y):
    return (torch.from_numpy(X).unsqueeze(1).to(device),
            torch.from_numpy(y).long().to(device))

Xtr, ytr = a_gpu(X_full[tr_idx], y_full[tr_idx])
Xva, yva = a_gpu(X_full[va_idx], y_full[va_idx])
Xte, yte = a_gpu(X_test, y_test)
MEAN = Xtr.float().mean().item() / 255
STD = Xtr.float().std().item() / 255

print(f'Clases: {CLASES}')
print(f'Entrenamiento: {len(ytr)} | Validación: {len(yva)} | Prueba: {len(yte)} '
      f'| carga en {time.time() - t0:.0f} s')
print(f'Media = {MEAN:.4f}, desviación = {STD:.4f}')

## 4. Modelo: ResNet-18 sin preentrenar

In [ ]:
def crear_modelo():
    m = torchvision.models.resnet18(weights=None, num_classes=len(CLASES))
    m.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)  # 1 canal
    return m.to(device).to(memory_format=torch.channels_last)

print(f'Parámetros: {sum(p.numel() for p in crear_modelo().parameters()) / 1e6:.2f} M')

## 5. Funciones de entrenamiento y evaluación

In [ ]:
criterion = nn.CrossEntropyLoss()

def fijar_semilla(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def preparar(xb, aumentar):
    x = (xb.float() / 255 - MEAN) / STD
    if aumentar:
        flip = torch.rand(x.size(0), device=x.device) < 0.5
        x = torch.where(flip[:, None, None, None], x.flip(3), x)
        pad = 8
        x = F.pad(x, (pad,) * 4, mode='reflect')
        i, j = torch.randint(0, 2 * pad + 1, (2,)).tolist()
        x = x[:, :, i:i + IMG_SIZE, j:j + IMG_SIZE]
    return x.contiguous(memory_format=torch.channels_last)

@torch.no_grad()
def evaluar(model, X, y):
    model.eval()
    loss_sum, correct = 0.0, 0
    for k in range(0, len(y), 256):
        xb, yb = preparar(X[k:k + 256], False), y[k:k + 256]
        with torch.autocast(device.type, enabled=USE_AMP):
            out = model(xb)
        loss_sum += criterion(out.float(), yb).item() * len(yb)
        correct += (out.argmax(1) == yb).sum().item()
    return loss_sum / len(y), correct / len(y)

def sincronizar():
    if device.type == 'cuda':
        torch.cuda.synchronize()

def entrenar(nombre, lr, seed, epochs=EPOCHS, verbose=True):
    # Entrena un optimizador con una semilla; devuelve historial por época y resumen
    fijar_semilla(seed)
    model = crear_modelo()
    optimizer = OPTIMIZADORES[nombre](model.parameters(), lr)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    gen = torch.Generator().manual_seed(seed)
    hist, best_acc, best_state, diverge = [], -1.0, None, False

    for ep in range(1, epochs + 1):
        model.train()
        sincronizar(); t0 = time.time()
        perm = torch.randperm(len(ytr), generator=gen).to(device)
        loss_sum, correct = 0.0, 0
        for k in range(0, len(perm), BATCH_SIZE):
            idx = perm[k:k + BATCH_SIZE]
            xb, yb = preparar(Xtr[idx], True), ytr[idx]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device.type, enabled=USE_AMP):
                out = model(xb)
            loss = criterion(out.float(), yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item()
        sincronizar(); t_train = time.time() - t0

        tr_loss, tr_acc = loss_sum / len(ytr), correct / len(ytr)
        if not math.isfinite(tr_loss):
            diverge = True
            if verbose: print(f'  época {ep}: DIVERGE (pérdida no finita)')
            break
        va_loss, va_acc = evaluar(model, Xva, yva)
        hist.append(dict(optimizador=nombre, semilla=seed, epoca=ep, lr=lr,
                         train_loss=tr_loss, train_acc=tr_acc, val_loss=va_loss,
                         val_acc=va_acc, tiempo_epoca=t_train))
        if va_acc > best_acc:
            best_acc, best_ep = va_acc, ep
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        if verbose:
            print(f'  época {ep:2d} | train loss {tr_loss:.4f} acc {tr_acc*100:5.2f}% | '
                  f'val loss {va_loss:.4f} acc {va_acc*100:5.2f}% | {t_train:.1f} s')

    if best_state is not None:
        model.load_state_dict(best_state)
        te_loss, te_acc = evaluar(model, Xte, yte)  # prueba: una sola vez, con la mejor época
    else:
        best_ep, te_loss, te_acc = 0, float('nan'), float('nan')

    ep90 = next((h['epoca'] for h in hist if h['val_acc'] >= 0.90), None)
    resumen = dict(optimizador=nombre, semilla=seed, lr=lr, diverge=diverge,
                   mejor_epoca=best_ep, mejor_val_acc=best_acc if best_state else float('nan'),
                   test_loss=te_loss, test_acc=te_acc,
                   train_loss_final=hist[-1]['train_loss'] if hist else float('nan'),
                   epoca_val90=ep90 if ep90 else float('nan'),
                   tiempo_epoca=float(np.mean([h['tiempo_epoca'] for h in hist])) if hist else float('nan'))
    return hist, resumen

## 6. *LR range test* (Smith, 2017)

Cada optimizador no funciona bien con la misma tasa de aprendizaje (por ejemplo, SGD suele necesitar
η ≈ 10⁻¹ y Adam η ≈ 10⁻³), así que usar una η común sería injusto. El *LR range test* entrena unas
iteraciones aumentando η exponencialmente desde 10⁻⁶ hasta 10 y registra la pérdida (suavizada).
Se elige **η = η<sub>mín</sub> / 10**, donde η<sub>mín</sub> es la tasa con la menor pérdida: un orden de
magnitud por debajo del punto en el que el entrenamiento empieza a volverse inestable. Se ignora el primer
10 % de iteraciones, donde la pérdida suavizada todavía es ruidosa.

In [ ]:
def lr_range_test(nombre, lr_min=1e-6, lr_max=10.0, iters=RANGE_ITERS, seed=0, beta=0.98):
    fijar_semilla(seed)
    model = crear_modelo(); model.train()
    optimizer = OPTIMIZADORES[nombre](model.parameters(), lr_min)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    gen = torch.Generator().manual_seed(seed)
    mult = (lr_max / lr_min) ** (1 / (iters - 1))
    lr, avg, best, lrs, losses, it = lr_min, 0.0, math.inf, [], [], 0
    while it < iters:
        perm = torch.randperm(len(ytr), generator=gen).to(device)
        for k in range(0, len(perm) - BATCH_SIZE + 1, BATCH_SIZE):
            for g in optimizer.param_groups:
                g['lr'] = lr
            idx = perm[k:k + BATCH_SIZE]
            xb, yb = preparar(Xtr[idx], True), ytr[idx]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device.type, enabled=USE_AMP):
                loss = criterion(model(xb).float(), yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
            l = loss.item()
            if not math.isfinite(l):
                return np.array(lrs), np.array(losses)
            avg = beta * avg + (1 - beta) * l
            smooth = avg / (1 - beta ** (it + 1))
            lrs.append(lr); losses.append(smooth)
            best = min(best, smooth)
            if smooth > 4 * best and it > 10:
                return np.array(lrs), np.array(losses)
            lr *= mult; it += 1
            if it >= iters:
                break
    return np.array(lrs), np.array(losses)

ruta_lr = os.path.join(OUT, 'lr_elegidos.json')
if os.path.exists(ruta_lr):
    LR = json.load(open(ruta_lr))
    print('Tasas de aprendizaje leídas de', ruta_lr)
    curvas = None
else:
    LR, curvas = {}, {}
    for nombre in OPTIMIZADORES:
        lrs, losses = lr_range_test(nombre)
        curvas[nombre] = (lrs, losses)
        skip = len(losses) // 10                     # primeras iteraciones: pérdida aún ruidosa
        lr_min_loss = lrs[skip + np.argmin(losses[skip:])]
        LR[nombre] = float(lr_min_loss / 10)
        print(f'{nombre:9s} η_mín = {lr_min_loss:.2e}  →  η elegida = {LR[nombre]:.2e}')
        if lr_min_loss < 1e-4:
            print(f'   AVISO: el mínimo de {nombre} está en la zona plana inicial; revisa la gráfica '
                  f'y, si hace falta, fija su η a mano más abajo.')
    json.dump(LR, open(ruta_lr, 'w'), indent=1)

    plt.figure(figsize=(9, 5))
    for nombre, (lrs, losses) in curvas.items():
        plt.semilogx(lrs, losses, color=COLORES[nombre], label=nombre)
        plt.axvline(LR[nombre], color=COLORES[nombre], ls=':', lw=1)
    plt.ylim(top=min(4, plt.ylim()[1]))
    plt.xlabel('Tasa de aprendizaje η (escala log)'); plt.ylabel('Pérdida de entrenamiento (suavizada)')
    plt.title('LR range test (líneas punteadas: η elegida)')
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(OUT, 'lr_range_test.png'), dpi=130); plt.show()

# Si quieres fijar a mano alguna tasa, descomenta y ajusta, p. ej.:
# LR['SGD'] = 0.05
display(pd.Series(LR, name='η elegida').map('{:.2e}'.format).to_frame())

## 7. Comparación: 7 optimizadores × 3 semillas

Cada entrenamiento guarda sus resultados en `resultados_semana2/` al terminar; si el entorno se
desconecta, al volver a ejecutar se saltan los que ya están hechos.

In [ ]:
ruta_hist = os.path.join(OUT, 'historial.csv')
ruta_res = os.path.join(OUT, 'resumen.csv')
hechos = set()
if os.path.exists(ruta_res):
    hechos = set(map(tuple, pd.read_csv(ruta_res)[['optimizador', 'semilla']].values.tolist()))

total = len(OPTIMIZADORES) * len(SEEDS)
n = 0
t_global = time.time()
for nombre in OPTIMIZADORES:
    for seed in SEEDS:
        n += 1
        if (nombre, seed) in hechos:
            print(f'[{n}/{total}] {nombre} semilla {seed}: ya hecho, se omite')
            continue
        print(f'[{n}/{total}] {nombre} (η = {LR[nombre]:.2e}) semilla {seed}')
        hist, res = entrenar(nombre, LR[nombre], seed)
        pd.DataFrame(hist).to_csv(ruta_hist, mode='a', header=not os.path.exists(ruta_hist), index=False)
        pd.DataFrame([res]).to_csv(ruta_res, mode='a', header=not os.path.exists(ruta_res), index=False)
        print(f'  → mejor val {res["mejor_val_acc"]*100:.2f}% (época {res["mejor_epoca"]}) | '
              f'test {res["test_acc"]*100:.2f}%')
print(f'\nTiempo total: {(time.time() - t_global) / 60:.1f} min')

## 8. Curvas de aprendizaje

Media de las semillas por época; la banda sombreada es ± una desviación estándar.

In [ ]:
H = pd.read_csv(ruta_hist)
agg = H.groupby(['optimizador', 'epoca']).agg(['mean', 'std']).reset_index()

paneles = [('train_loss', 'Pérdida de entrenamiento', True), ('val_loss', 'Pérdida de validación', True),
           ('train_acc', 'Exactitud de entrenamiento', False), ('val_acc', 'Exactitud de validación', False)]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (col, titulo, logy) in zip(axes.ravel(), paneles):
    for nombre in OPTIMIZADORES:
        d = agg[agg['optimizador'] == nombre]
        if d.empty:
            continue
        m, s = d[(col, 'mean')], d[(col, 'std')].fillna(0)
        if col.endswith('acc'):
            m, s = m * 100, s * 100
        ax.plot(d['epoca'], m, color=COLORES[nombre], label=nombre, lw=1.8)
        ax.fill_between(d['epoca'], m - s, m + s, color=COLORES[nombre], alpha=0.15)
    ax.set_title(titulo); ax.set_xlabel('Época'); ax.grid(alpha=0.3)
    ax.set_ylabel('Pérdida' if 'loss' in col else 'Exactitud (%)')
    if logy:
        ax.set_yscale('log')
axes[0, 0].legend()
fig.suptitle(f'ResNet-18 desde cero: comparación de optimizadores (media ± desv. de {len(SEEDS)} semillas)')
fig.tight_layout()
fig.savefig(os.path.join(OUT, 'curvas_optimizadores.png'), dpi=130)
plt.show()

## 9. Tabla de resultados

- **Mejor val. acc.:** exactitud de validación en la mejor época.
- **Test acc.:** exactitud en prueba del modelo de la mejor época de validación.
- **Época val ≥ 90 %:** primera época en que la validación alcanza el 90 % (velocidad de convergencia).
- **Tiempo/época:** solo entrenamiento, en segundos.

In [ ]:
R = pd.read_csv(ruta_res)

def ms(serie, pct=False, dec=2):
    v = serie.dropna() * (100 if pct else 1)
    if v.empty:
        return '—'
    return f'{v.mean():.{dec}f} ± {v.std(ddof=1) if len(v) > 1 else 0:.{dec}f}'

filas = []
for nombre in OPTIMIZADORES:
    r = R[R['optimizador'] == nombre]
    if r.empty:
        continue
    filas.append({
        'Optimizador': nombre,
        'η': f"{r['lr'].iloc[0]:.1e}",
        'Mejor val. acc. (%)': ms(r['mejor_val_acc'], True),
        'Test acc. (%)': ms(r['test_acc'], True),
        'Pérdida train final': ms(r['train_loss_final'], dec=3),
        'Época val ≥ 90 %': ms(r['epoca_val90'], dec=1),
        'Tiempo/época (s)': ms(r['tiempo_epoca'], dec=1),
        'Divergencias': f"{int(r['diverge'].sum())}/{len(r)}",
    })
tabla = pd.DataFrame(filas)
tabla.to_csv(os.path.join(OUT, 'tabla_resultados.csv'), index=False)
display(tabla)

# Gráfica de barras: exactitud en prueba (media ± desviación)
g = R.groupby('optimizador')['test_acc'].agg(['mean', 'std']).reindex(
    [o for o in OPTIMIZADORES if o in set(R['optimizador'])]) * 100
plt.figure(figsize=(9, 4.5))
plt.bar(g.index, g['mean'], yerr=g['std'].fillna(0), capsize=5,
        color=[COLORES[o] for o in g.index])
plt.ylabel('Exactitud en prueba (%)')
plt.ylim(max(0, g['mean'].min() - 15), 100)
plt.title(f'Exactitud en prueba por optimizador (media ± desv., {len(SEEDS)} semillas)')
plt.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(OUT, 'barras_test_acc.png'), dpi=130)
plt.show()

## 10. Descargar resultados

Descarga un `.zip` con el código, las figuras y las tablas para subirlo a Classroom.

In [ ]:
!zip -qr semana2.zip optimizadores.py {OUT}
from google.colab import files
files.download('semana2.zip')